# 🗿 modelo3d — De foto a modelo 3D imprimible

Convertí una foto (o tres vistas del mismo objeto) en un archivo **STL listo para imprimir**, sin saber nada de programación.

## Qué necesitás
- Una cuenta de Google (gratis).
- Una foto del objeto: buena luz, fondo liso, un solo objeto centrado.

## Cuánto tarda
- **Primera vez:** 5–8 minutos de instalación automática (solo una vez por sesión).
- **Cada modelo:** entre 30 segundos y 2 minutos.

## Antes de empezar
1. Hacé clic en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU → Guardar**.
2. Ejecutá la celda de instalación de abajo y esperá el mensaje ✅.
3. La aplicación va a aparecer al final de la página.

⚠️ **Importante:** cuando la sesión de Colab se cierre, los archivos se borran. Descargá tu STL apenas lo generes.


In [ ]:
# --- Instalación (ejecutar primero) ---
import sys

if not __import__("torch").cuda.is_available():
    raise RuntimeError(
        "GPU no activada. Andá a 'Entorno de ejecución → Cambiar tipo de entorno "
        "de ejecución → T4 GPU', guardá, y volvé a ejecutar esta celda."
    )

print("✅ GPU detectada:", __import__("torch").cuda.get_device_name(0))

REPO_URL = "https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git"
REPO_DIR = "/content/Hunyuan3D-2"
MODEL_SINGLE = ("tencent/Hunyuan3D-2mini", "hunyuan3d-dit-v2-mini-turbo")
MODEL_MULTI = ("tencent/Hunyuan3D-2mv", "hunyuan3d-dit-v2-mv")
HF_REVISIONS = {
    "mini": "f90a0f7df7d5e6f71109cf333f6a95a0ae3194a6",
    "mv": "3a761b539b29fe4ff64714813aa9560fd66f5de0",
}

import subprocess  # noqa: E402

subprocess.run(
    ["bash", "-lc",
     f'test -d {REPO_DIR} || git clone --depth 1 {REPO_URL} {REPO_DIR}'],
    check=True,
)
subprocess.run(
    ["bash", "-lc",
     f'pip install -q -r {REPO_DIR}/requirements.txt '
     'pyrembg trimesh pymeshfix manifold3d gradio'],
    check=True,
)
sys.path.insert(0, REPO_DIR)

from huggingface_hub import snapshot_download  # noqa: E402

print("⬇️ Descargando modelo de una foto (~1 GB)...")
snapshot_download(
    repo_id=MODEL_SINGLE[0],
    revision=HF_REVISIONS["mini"],
    allow_patterns=[f"{MODEL_SINGLE[1]}/*"],
)
print("⬇️ Descargando modelo multivista (~2 GB)...")
snapshot_download(
    repo_id=MODEL_MULTI[0],
    revision=HF_REVISIONS["mv"],
    allow_patterns=[f"{MODEL_MULTI[1]}/*"],
)

_ENGINES = None


def ensure_engines():
    """Carga diferida de los pipelines (solo forma, sin texturas)."""
    global _ENGINES
    if _ENGINES is None:
        from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline

        mini = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            MODEL_SINGLE[0], subfolder=MODEL_SINGLE[1]
        )
        mini.to("cuda")
        try:
            mv = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
                MODEL_MULTI[0], subfolder=MODEL_MULTI[1]
            )
            mv.to("cuda")
        except Exception:
            print("⚠️ Motor multivista no disponible; se usará el de una foto.")
            mv = None
        _ENGINES = (mini, mv)
    return _ENGINES


print("✅ Instalación lista. Ejecutá la celda de abajo para abrir la app.")


In [ ]:
# --- Configuración general ---
SIZE_PRESETS_MM = {"10cm": 100, "15cm": 150}
DEFAULT_PRESET = "10cm"


def resolve_size_preset(preset: str, custom_mm: int | None) -> int:
    """Devuelve la altura objetivo en milímetros."""
    if preset in SIZE_PRESETS_MM:
        return SIZE_PRESETS_MM[preset]
    if preset == "custom":
        if not custom_mm or custom_mm <= 0:
            raise ValueError("Ingresá un alto en milímetros válido (mayor a 0).")
        return int(custom_mm)
    raise ValueError(f"Tamaño desconocido: {preset}")

In [ ]:
# --- Validación de fotos y mensajes ---
import numpy as np
from PIL import Image

MIN_SIDE_PX = 256

ERRORS_ES = {
    "too_small": "La foto es muy chica. Usá una imagen de al menos 256 píxeles por lado.",
    "unreadable": "No pudimos leer la imagen. Probá con otro archivo JPG o PNG.",
    "no_object": "No detectamos ningún objeto en la foto. Revisá que el objeto se vea completo y con buen contraste contra el fondo.",
    "bad_cutout": "El recorte del objeto quedó raro. Sacá la foto con el objeto centrado sobre un fondo liso, sin manos y sin que se corte con el borde.",
}

FALLBACK_ERROR_ES = (
    "Algo salió mal generando el modelo. Probá de nuevo; si sigue fallando, "
    "probá con otra foto."
)


def _to_rgb(img: Image.Image) -> Image.Image:
    return img.convert("RGB") if img.mode != "RGB" else img


def validate_image(img: Image.Image) -> None:
    try:
        img = _to_rgb(img)
        w, h = img.size
    except Exception as exc:
        raise ValueError(ERRORS_ES["unreadable"]) from exc
    if w < MIN_SIDE_PX or h < MIN_SIDE_PX:
        raise ValueError(ERRORS_ES["too_small"])


def mask_fraction(mask: np.ndarray) -> float:
    return float(np.count_nonzero(mask)) / float(mask.size)


def check_mask_sane(fraction: float) -> None:
    if fraction < 0.01:
        raise ValueError(ERRORS_ES["no_object"])
    if fraction > 0.90:
        raise ValueError(ERRORS_ES["bad_cutout"])


def friendly_error(exc: Exception) -> str:
    if isinstance(exc, ValueError) and str(exc) in ERRORS_ES.values():
        return str(exc)
    return FALLBACK_ERROR_ES


In [ ]:
# --- Núcleo geométrico: reparar, escalar, base, exportar ---
import numpy as np
import trimesh
import trimesh.boolean

BASE_HEIGHT_MM = 3.0
BASE_MARGIN = 0.95  # radio del pedestal relativo al alcance XY del modelo


def repair_mesh(mesh: trimesh.Trimesh) -> trimesh.Trimesh:
    mesh = mesh.copy()
    mesh.merge_vertices()
    mesh.update_faces(mesh.nondegenerate_faces())
    mesh.update_faces(mesh.unique_faces())
    mesh.remove_unreferenced_vertices()
    if not mesh.is_watertight:
        try:
            import pymeshfix

            verts = np.asarray(mesh.vertices, dtype=np.float64).copy()
            faces = np.asarray(mesh.faces, dtype=np.int32).copy()
            fixed = pymeshfix.clean_from_arrays(verts, faces)
            mesh = trimesh.Trimesh(fixed[0], fixed[1], process=False)
        except Exception as exc:
            raise ValueError(
                "El modelo salió con agujeros que no pudimos reparar. "
                "Probá generar de nuevo con otra foto."
            ) from exc
    if mesh.volume < 0:
        mesh.invert()
    return mesh


def normalize_mesh(
    mesh: trimesh.Trimesh, target_height_mm: float
) -> trimesh.Trimesh:
    m = mesh.copy()
    height = float(m.extents[2])
    if height <= 0:
        raise ValueError("La geometría generada es plana e inválida.")
    m.apply_scale(target_height_mm / height)
    m.apply_translation(-m.bounds[0])          # apoyar en Z=0
    center_xy = m.bounds.mean(axis=0)[:2]
    m.apply_translation([-center_xy[0], -center_xy[1], 0])
    return m


def _pedestal(radius_mm: float, height_mm: float) -> trimesh.Trimesh:
    cyl = trimesh.creation.cylinder(
        radius=radius_mm, height=height_mm, sections=64
    )
    cyl.apply_translation([0, 0, height_mm / 2.0])
    return cyl


def add_flat_base(
    mesh: trimesh.Trimesh, height_mm: float = BASE_HEIGHT_MM
) -> trimesh.Trimesh:
    xy_span = float(max(mesh.extents[0], mesh.extents[1]))
    pedestal = _pedestal(xy_span * BASE_MARGIN * 0.5, height_mm)
    merged = trimesh.boolean.union([mesh, pedestal], engine="manifold")
    return merged


def export_stl(mesh: trimesh.Trimesh, path: str) -> str:
    mesh.export(path, file_type="stl")
    return path


def verify_stl(
    path: str, target_height_mm: float, tol_mm: float = 0.5
) -> trimesh.Trimesh:
    reloaded = trimesh.load(path, force="mesh")
    if not reloaded.is_watertight:
        raise ValueError("El STL exportado tiene agujeros.")
    if reloaded.volume <= 0:
        raise ValueError("El STL exportado está vacío.")
    height = float(reloaded.extents[2])
    if abs(height - target_height_mm) > tol_mm:
        raise ValueError(
            f"El STL mide {height:.1f} mm de alto en vez de {target_height_mm:.1f} mm."
        )
    return reloaded
